In [2]:
from langchain_core.documents import Document

# Ingestion Pipeline

### Data to Documents

In [13]:
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

In [18]:
def load_all_pdf():
    folder_path = "data/pdf"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path): #get name of all docs in folder
        if filename.lower().endswith(".pdf"):
            #complete file path
            pdf_path = os.path.join(folder_path,filename)

            loader = PyPDFLoader(pdf_path)
            document = loader.load()
            
            all_docs.extend(document)
            num_docs += 1
    print("total pdfs:",num_docs)
    print("total pages:",len(all_docs))
    return all_docs

In [19]:
all_pdf_documents = load_all_pdf()

total pdfs: 2
total pages: 32


In [22]:
type(all_pdf_documents[0])

langchain_core.documents.base.Document

# Chunking

In [24]:
# !pip install langchain_text_splitters

In [28]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_doc(documents,chunk_size=500,chunk_overlap=50):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [29]:
chunks = split_doc(all_pdf_documents)

In [30]:
len(chunks)

321

# Embeddings

In [31]:
from sentence_transformers import SentenceTransformer

In [34]:
class EmbeddingManager: #embedding model
    def __init__(self,model_name = "all-MiniLM-L6-v2"):
        
        self.model_name = model_name
        print("loading model....:",self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimension:",self.model.get_sentence_embedding_dimension())


    def generate_embeddings(self,text):
        embeddings = self.model.encode(text,show_progress_bar = True)
        print("embeddings shape:",embeddings.shape)
        return embeddings

In [35]:
embedding_manager = EmbeddingManager()

loading model....: all-MiniLM-L6-v2


Loading weights: 100%|███████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 8507.05it/s]


embedding dimension: 384


/tmp/ipykernel_383050/357030263.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimension:",self.model.get_sentence_embedding_dimension())


# Vector store

In [36]:
import chromadb
import uuid #create index for values

In [37]:
class VectorStoreManger:
    def __init__(self,persist_directory="data/vector_store",collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self.initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory,exist_ok=True)

        #create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        #create the collection
        self.collection = self.client.get_or_create_collection(
            name = self.collection_name,
            metadata={"description":"This is a vector store collection for pdf embeddings in RAG"}
        )

        print("initalized the vector store with collection:",self.collection_name)
        print("docs in collection:",self.collection.count())

In [39]:
vector_store = VectorStoreManager()

NameError: name 'VectorStoreManager' is not defined